In [1]:
pip install razdel

# Эта строка устанавливает библиотеку razdel, которая используется для токенизации текста на русском языке. Токенизация — это процесс разбиения текста на отдельные слова или токены.

In [ ]:
import os # для работы с файловой системой
import re # для работы с регулярными выражениями (используется для поиска и замены текста).
import string # для работы со строками, например, для удаления пунктуации
import unicodedata #д ля нормализации текста (приведение символов к единому формату)
import nltk # библиотека для обработки естественного языка

from bs4 import BeautifulSoup # для работы с HTML/XML (хотя в данном коде он не используется).
from razdel import tokenize # для токенизации текста
from nltk.corpus import stopwords, wordnet # для работы со стоп-словами и лемматизацией
from nltk.stem import WordNetLemmatizer # для лемматизации слов (приведение слов к их базовой форме)

In [ ]:
nltk.download('stopwords') # список стоп-слов (слов, которые часто удаляются из текста, так как они не несут смысловой нагрузки, например, "и", "в", "на").
nltk.download('wordnet') # лексическая база данных для английского языка, используемая для лемматизации.
nltk.download('omw-1.4') # Open Multilingual Wordnet, который расширяет возможности WordNet на другие языки.
nltk.download('averaged_perceptron_tagger') # модель для определения части речи слова, что важно для лемматизации.

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng') # загружает модель для определения части речи слова на английском языке. Это нужно для корректной работы лемматизатора

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
lemmatizer = WordNetLemmatizer() # это объект, который будет использоваться для лемматизации слов (приведения слов к их базовой форме).
stop_words = set(stopwords.words('russian')) # это набор стоп-слов на русском языке, которые будут удалены из текста.

In [ ]:
def clean_text(text):
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation))) # заменяет все знаки пунктуации на пробелы.
    text = re.sub(r'\s+', ' ', text).strip() # удаляет лишние пробелы и обрезает пробелы в начале и конце строки.
    return text

In [ ]:
def normalize_text(text):
    text = text.lower() # приводит текст к нижнему регистру.
    text = unicodedata.normalize('NFKC', text) # нормализует текст, приводя символы к единому формату.
    # re.sub ниже заменяют всевозможные символы на пробелы
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[@#$%^&*]', ' ', text)
    text = re.sub(r'[-—]', ' ', text)
    text = re.sub(r'[«»]', ' ', text)
    text = re.sub(r'[–]', ' ', text)
    text = re.sub(r'\\[a-zA-Z0-9]+', ' ', text)
    text = re.sub(r'\bet al\b rc\b gmr\b', ' ', text)
    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)
    text = re.sub(r'[„]', ' ', text)
    text = re.sub(r'[“]', ' ', text)
    text = re.sub(r'[−]', ' ', text)
    text = re.sub(r'[≈]', ' ', text)
    text = re.sub(r'[α]', ' ', text)
    text = re.sub(r'\bet al\b|\brc\b|\bgmr\b|\bct\b|\bc⋅gmr\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip() # удаляются лишние пробелы
    return text


In [ ]:
def preprocess_text(text):
    text = clean_text(text)
    text = normalize_text(text)

    # Токенизация с использованием razdel
    tokens = [token.text for token in tokenize(text)] # токенизирует текст с помощью библиотеки razdel.
    print("После токенизации:", tokens[:20], '...')

    filtered_tokens = [word for word in tokens if word not in stop_words] # удаляет стоп-слова из списка токенов.
    print("После удаления стоп-слов:", filtered_tokens[:20], '...')

    lemmatized_tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in filtered_tokens] # лемматизирует слова (приводит их к базовой форме).
    print("После лемматизации:", lemmatized_tokens[:20], '...')

    return {
        "cleaned_text": " ".join(lemmatized_tokens),
        "tokens": lemmatized_tokens
    }


In [ ]:
file_paths = [
    "/content/Текст_4.txt",
    "/content/Текст_3.txt",
    "/content/Текст_2.txt",
    "/content/Текст_1.txt"
]

In [ ]:
for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
        processed_data = preprocess_text(text)

    # Сохранение результатов в новые файлы
    output_path = file_path.replace(".txt", "_processed.txt")
    with open(output_path, 'w', encoding='utf-8') as output_file:
        output_file.write(processed_data['cleaned_text'])

    print("Финальные токены:", processed_data['tokens'][:20], '...')

После токенизации: ['не', 'так', 'уж', 'часто', 'бывает', 'что', 'две', 'конкурирующие', 'гипотезы', 'сосуществуют', 'десятки', 'лет', 'сменяя', 'друг', 'друга', 'в', 'борьбе', 'за', 'звание', 'общепринятой'] ...
После удаления стоп-слов: ['часто', 'бывает', 'две', 'конкурирующие', 'гипотезы', 'сосуществуют', 'десятки', 'лет', 'сменяя', 'друг', 'друга', 'борьбе', 'звание', 'общепринятой', 'именно', 'такая', 'ситуация', 'сложилась', 'космологии', 'начиная'] ...
После лемматизации: ['часто', 'бывает', 'две', 'конкурирующие', 'гипотезы', 'сосуществуют', 'десятки', 'лет', 'сменяя', 'друг', 'друга', 'борьбе', 'звание', 'общепринятой', 'именно', 'такая', 'ситуация', 'сложилась', 'космологии', 'начиная'] ...
Финальные токены: ['часто', 'бывает', 'две', 'конкурирующие', 'гипотезы', 'сосуществуют', 'десятки', 'лет', 'сменяя', 'друг', 'друга', 'борьбе', 'звание', 'общепринятой', 'именно', 'такая', 'ситуация', 'сложилась', 'космологии', 'начиная'] ...
После токенизации: ['американские', 'биологи'